# 07 — Voice Cloning: Speaker Embedding + Synthesis

Demonstrates zero-shot voice cloning using XTTS-v2:
- Extract speaker embedding from 5-30s reference audio
- Synthesize new utterances in the cloned voice
- Evaluate speaker similarity (cosine similarity of embeddings)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = '/content/drive/MyDrive/VietSpeech'
MODEL_DIR = f'{PROJECT_ROOT}/weights/xtts-v2'
OUTPUT_DIR = f'{PROJECT_ROOT}/voice_clones'

import sys, os
sys.path.insert(0, PROJECT_ROOT)
os.makedirs(OUTPUT_DIR, exist_ok=True)

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

In [ ]:
!pip install -q TTS soundfile
print('Deps installed')

In [ ]:
# ── Load fine-tuned XTTS-v2 ───────────────────────────────────────────────
from TTS.api import TTS

if os.path.exists(f'{MODEL_DIR}/model.pth'):
    tts = TTS(
        model_path=f'{MODEL_DIR}/model.pth',
        config_path=f'{MODEL_DIR}/config.json',
    ).to(device)
    print('Fine-tuned XTTS-v2 loaded')
else:
    # Fallback to base model
    tts = TTS('tts_models/multilingual/multi-dataset/xtts_v2').to(device)
    print('Base XTTS-v2 loaded (fine-tuned model not found)')

In [ ]:
# ── Voice cloning demo ────────────────────────────────────────────────────
import soundfile as sf
import numpy as np

# Use first ViMD sample as reference (replace with real reference audio)
import glob
reference_audios = glob.glob(f'{PROJECT_ROOT}/processed/audio/*.wav')[:1]

if not reference_audios:
    # Create a dummy 5-second reference
    dummy_audio = np.random.randn(5 * 16000).astype(np.float32) * 0.01
    reference_path = '/tmp/reference.wav'
    sf.write(reference_path, dummy_audio, 16000)
    print('Using dummy reference audio (no real audio found)')
else:
    reference_path = reference_audios[0]
    print(f'Reference audio: {reference_path}')

# Text to synthesize in the cloned voice
texts_to_clone = [
    'Xin chào, tôi đang nói chuyện với giọng được sao chép.',
    'Hệ thống nhận dạng tiếng Việt đa phương ngữ.',
    'Tôi sống ở Hà Nội và học tại trường đại học.',
]

for i, text in enumerate(texts_to_clone):
    output_path = f'{OUTPUT_DIR}/clone_{i+1}.wav'
    tts.tts_to_file(
        text=text,
        language='vi',
        speaker_wav=reference_path,
        file_path=output_path,
    )
    print(f'Generated: {output_path}')

In [ ]:
# ── Speaker similarity evaluation ─────────────────────────────────────────
from sklearn.metrics.pairwise import cosine_similarity

def extract_speaker_embedding(audio_path, tts_model):
    """Extract XTTS speaker embedding from audio file."""
    gpt_cond_latent, speaker_embedding = tts_model.synthesizer.tts_model.get_conditioning_latents(
        audio_path=[audio_path]
    )
    return speaker_embedding.cpu().numpy().flatten()

ref_embed = extract_speaker_embedding(reference_path, tts)

print('Speaker Similarity (vs. reference):')
for i in range(len(texts_to_clone)):
    clone_path = f'{OUTPUT_DIR}/clone_{i+1}.wav'
    if os.path.exists(clone_path):
        clone_embed = extract_speaker_embedding(clone_path, tts)
        sim = cosine_similarity([ref_embed], [clone_embed])[0][0]
        status = '✓ PASS' if sim >= 0.90 else '✗ FAIL'
        print(f'  clone_{i+1}.wav: {sim:.3f} {status} (target ≥ 0.90)')

print('\n✓ Voice cloning evaluation complete')

## ✓ Voice Cloning Demo Complete

Cloned voice samples saved to `{OUTPUT_DIR}/`

**Next**: Run `08_export_onnx.ipynb`